# Project 2 - Deep Learning Based Arabic Audio Understanding and Retrieval System

## Student Credentials:
- Name: Yousef Ibrahim Gomaa Mahmoud
- ID: 320210207

# Mounting to Drive (Copy & Shortcut)

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
from google.colab import auth
auth.authenticate_user()

import re
from googleapiclient.discovery import build

drive_api = build("drive", "v3")

def extract_drive_id(url: str):
    patterns = [
        r"/file/d/([a-zA-Z0-9_-]+)",
        r"/folders/([a-zA-Z0-9_-]+)",
        r"[?&]id=([a-zA-Z0-9_-]+)",
        r"/d/([a-zA-Z0-9_-]+)",
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    raise ValueError("Could not extract Drive ID from the link.")

SHARED_URL = "https://drive.google.com/drive/folders/0B28Rhpdi0XLMfldKUUN6ZlhCTmRwLUhaUTZBN3ZBMUFDcjRVWVV3TTFtVTZobUVhMFBGSzg"
TARGET_ID = extract_drive_id(SHARED_URL)

SHORTCUT_FOLDER_NAME = "ColabShared"
SHORTCUT_NAME = "shared_item"

In [3]:
# Create a folder in MyDrive to hold the shortcut
folder = drive_api.files().create(
    body={
        "name": SHORTCUT_FOLDER_NAME,
        "mimeType": "application/vnd.google-apps.folder",
        "parents": ["root"],
    },
    fields="id, name"
).execute()

shortcut_folder_id = folder["id"]
print("Created folder:", folder)

Created folder: {'id': '1Vs7Hlrnf5IR-GfIjiSsBviLdiOfCjgbf', 'name': 'ColabShared'}


In [4]:
# Create the shortcut inside that folder
shortcut = drive_api.files().create(
    body={
        "name": SHORTCUT_NAME,
        "mimeType": "application/vnd.google-apps.shortcut",
        "parents": [shortcut_folder_id],
        "shortcutDetails": {
            "targetId": TARGET_ID
        }
    },
    fields="id, name, shortcutDetails"
).execute()

print("Created shortcut:", shortcut)

Created shortcut: {'id': '1j5z7sCiI57JXTbZODqoP3Cxuu70hAeOx', 'name': 'shared_item', 'shortcutDetails': {'targetId': '0B28Rhpdi0XLMfldKUUN6ZlhCTmRwLUhaUTZBN3ZBMUFDcjRVWVV3TTFtVTZobUVhMFBGSzg', 'targetMimeType': 'application/vnd.google-apps.folder', 'targetResourceKey': '0-x-kF6vPagWEyH9kjdVSl_g'}}


In [6]:
import os

shortcut_path = f"/content/drive/MyDrive/{SHORTCUT_FOLDER_NAME}/{SHORTCUT_NAME}"
print("Shortcut path:", shortcut_path)
print("Exists:", os.path.exists(shortcut_path))

!ls -lah "/content/drive/MyDrive/ColabShared"

Shortcut path: /content/drive/MyDrive/ColabShared/shared_item
Exists: True
total 0
lrw------- 1 root root 0 Apr  4 21:08 shared_item -> /content/drive/.shortcut-targets-by-id/0B28Rhpdi0XLMfldKUUN6ZlhCTmRwLUhaUTZBN3ZBMUFDcjRVWVV3TTFtVTZobUVhMFBGSzg/cut_clips9


In [7]:
DATA_PATH = "/content/drive/MyDrive/ColabShared/shared_item"

In [8]:
print("Contents:")
print(os.listdir(DATA_PATH))

Contents:
['ARA NORM 00025.wav', 'ARA NORM 00154.wav', 'ARA NORM 00162.wav', 'ARA NORM 00126.wav', 'ARA NORM 00128.wav', 'ARA NORM 00135.wav', 'ARA NORM 01584.wav', 'ARA NORM 00124.wav', 'ARA NORM 00350.wav', 'ARA NORM 00361.wav', 'ARA NORM 00127.wav', 'ARA NORM 00131.wav', 'ARA NORM 01941.wav', 'ARA NORM 00318.wav', 'ARA NORM 00009.wav', 'ARA NORM 00328.wav', 'ARA NORM 00640.wav', 'ARA NORM 00020.wav', 'ARA NORM 00093.wav', 'ARA NORM 00137.wav', 'ARA NORM 00075.wav', 'ARA NORM 00230.wav', 'ARA NORM 00524.wav', 'ARA NORM 00994.wav', 'ARA NORM 00103.wav', 'ARA NORM 00305.wav', 'ARA NORM 00272.wav', 'ARA NORM 00231.wav', 'ARA NORM 00329.wav', 'ARA NORM 00358.wav', 'ARA NORM 00088 (1).wav', 'ARA NORM 00113.wav', 'ARA NORM 00228.wav', 'ARA NORM 00326.wav', 'ARA NORM 00121.wav', 'ARA NORM 00345.wav', 'ARA NORM 00144.wav', 'ARA NORM 00208.wav', 'ARA NORM 00314.wav', 'ARA NORM 00290.wav', 'ARA NORM 00260.wav', 'ARA NORM 00140.wav', 'ARA NORM 00155.wav', 'ARA NORM 00188.wav', 'ARA NORM 00250.w

In [9]:
!pip -q install -U transformers accelerate librosa soundfile pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 91.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.


# Transcription

In [27]:
!pip -q install -U nemo_toolkit['asr'] huggingface_hub pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 109.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.


In [14]:
OUTPUT_CSV = "/content/drive/MyDrive/ColabShared/whisper_base_ar_first5000.csv"

In [15]:
import os
import re
import glob



MAX_FILES = 5000

def natural_key(path):
    name = os.path.basename(path)
    m = re.search(r'(\d+)(?=\.[^.]+$)', name)
    if m:
        prefix = name[:m.start(1)].strip()
        number = int(m.group(1))
        return (prefix, number, name)
    return (name, -1, name)

all_wavs = sorted(glob.glob(os.path.join(DATA_PATH, "*.wav")), key=natural_key)
selected_files = all_wavs[:MAX_FILES]
selected_names = [os.path.basename(p) for p in selected_files]

print("Selected:", len(selected_files))
print(selected_names[:10])

Selected: 5000
['ARA NORM 00001.wav', 'ARA NORM 00002.wav', 'ARA NORM 00003.wav', 'ARA NORM 00004.wav', 'ARA NORM 00005.wav', 'ARA NORM 00006.wav', 'ARA NORM 00007.wav', 'ARA NORM 00008.wav', 'ARA NORM 00009.wav', 'ARA NORM 00010.wav']


In [16]:
from datasets import Dataset, Audio

ds = Dataset.from_dict({
    "file_name": selected_names,
    "audio": selected_files,
})

# decode + resample on access
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

print(ds)
print(ds[0]["file_name"])
print(ds[0]["audio"]["sampling_rate"])

Dataset({
    features: ['file_name', 'audio'],
    num_rows: 5000
})
ARA NORM 00001.wav
16000


## Whisper Base (Supports Arabic)

In [28]:
import torch
from huggingface_hub import hf_hub_download
from nemo.collections.asr.models import ASRModel

repo_id = "NAMAA-Space/EgypTalk-ASR-v2"
filename = "asr-egyptian-nemo-v2.0.nemo"

device = "cuda" if torch.cuda.is_available() else "cpu"

nemo_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename
)

print("Downloaded to:", nemo_path)

model = ASRModel.restore_from(
    restore_path=nemo_path,
    map_location=device
)

model.eval()
if device == "cuda":
    model = model.to("cuda")

asr-egyptian-nemo-v2.0.nemo:   0%|          | 0.00/459M [00:00<?, ?B/s]

Downloaded to: /root/.cache/huggingface/hub/models--NAMAA-Space--EgypTalk-ASR-v2/snapshots/d23895cf2512efc62b5a34eafa8672b26c530953/asr-egyptian-nemo-v2.0.nemo
[NeMo I 2026-04-04 21:53:12 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-04-04 21:53:14 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: datasets/manifest_train.json
    sample_rate: 16000
    batch_size: 32
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 20
    min_duration: 0.5
    is_tarred: false
    tarred_audio_filepaths: false
    shuffle_n: 2048
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    use_start_end_token: true
    trim_silence: true
    
[NeMo W 2026-04-04 21:53:14 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: datasets/manifest_test.json
    sample_rate: 16000
    batch_size: 8
 

[NeMo I 2026-04-04 21:53:15 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-04-04 21:53:15 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-04-04 21:53:15 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-04-04 21:53:16 save_restore_connector:285] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /root/.cache/huggingface/hub/models--NAMAA-Space--EgypTalk-ASR-v2/snapshots/d23895cf2512efc62b5a34eafa8672b26c530953/asr-egyptian-nemo-v2.0.nemo.


In [29]:
import os
import re
import glob
import pandas as pd
from tqdm.auto import tqdm

OUTPUT_CSV = "/content/drive/MyDrive/ColabShared/transcript_first5000.csv"
MAX_FILES = 5000
BATCH_SIZE = 8

def natural_key(path):
    name = os.path.basename(path)
    m = re.search(r'(\d+)(?=\.[^.]+$)', name)
    if m:
        prefix = name[:m.start(1)].strip()
        number = int(m.group(1))
        return (prefix, number, name)
    return (name, -1, name)

all_wavs = sorted(glob.glob(os.path.join(DATA_PATH, "*.wav")), key=natural_key)
selected_files = all_wavs[:MAX_FILES]

rows = []

for i in tqdm(range(0, len(selected_files), BATCH_SIZE), desc="Transcribing"):
    batch_files = selected_files[i:i+BATCH_SIZE]
    batch_names = [os.path.basename(x) for x in batch_files]

    try:
        outputs = model.transcribe(batch_files, batch_size=len(batch_files))
        texts = [o.text if hasattr(o, "text") else str(o) for o in outputs]

        for fn, txt in zip(batch_names, texts):
            rows.append({
                "file_name": fn,
                "transcription": txt.strip()
            })
    except Exception as e:
        for fn in batch_names:
            rows.append({
                "file_name": fn,
                "transcription": "",
                "error": str(e)
            })

    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved to:", OUTPUT_CSV)

Transcribing:   0%|          | 0/625 [00:00<?, ?it/s]

[NeMo W 2026-04-04 21:53:44 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-04-04 21:53:45 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)

Transcribing: 0it [00:00, ?it/s]
Transcribing: 1it [00:03,  3.58s/it]
[NeMo W 2026-04-04 21:53:50 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-04-04 21:53:50 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impact

KeyboardInterrupt: 

In [30]:
df = pd.read_csv(OUTPUT_CSV)
print(df.shape)
df.head(10)

(1608, 2)


,file_name,transcription
0,ARA NORM 00001.wav,هل التطبيق الصيني المعروف باسم
1,ARA NORM 00002.wav,التيك توك
2,ARA NORM 00003.wav,هو أخطر تطبيق في العالم
3,ARA NORM 00004.wav,السؤال دا مهم جداً ولازم كلنا نفكر فيه
4,ARA NORM 00005.wav,هو في الحقيقه شاغل بال ناس كتير
5,ARA NORM 00006.wav,وبالأخص صناع القرار في الولايات المتحدة
6,ARA NORM 00007.wav,واللي بيعتبر التطبيق ده اخطر حاجه اخترعتها
7,ARA NORM 00008.wav,الصين على الإطلاق
8,ARA NORM 00009.wav,البنتاغون طلبت من افراد الجيش الامريكي
9,ARA NORM 00010.wav,إنهم يحذفوا بسرعة تطبيق تيك توك
